<div dir="ltr" align="left" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>Midterm, Part A: reruns after the epoch-cap change, and the new metrics for every run</p>
<p>A thin launcher: it clones the repository, installs it and runs train.py. All the code is in the repository.</p>
<p><a href="https://colab.research.google.com/github/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR/blob/main/colab_partA.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a></p>
<p>Why this notebook:</p>
<ul>
<li>With the old caps (30 epochs for linear_probe and finetune, 12 for the supervised encoder), 18 runs stopped while val_accuracy was still rising. The cap is now 100 for every classifier; EarlyStopping (patience 5) still ends each run sooner.</li>
<li>Classifiers now also log macro precision, recall, F1 and AUROC, and save the test confusion matrix. analysis saves the same for the k-NN.</li>
</ul>
<p>Nothing is deleted. A run that is trained again keeps its old results under a new name (results_cap30.json, ..._cap12).</p>
<p>Order of the cells:</p>
<ol>
<li>Clone the repository, or pull the latest code.</li>
<li>Install the requirements and check the GPU (L4).</li>
<li>Mount Google Drive.</li>
<li>Start TensorBoard.</li>
<li>The 22 runs the old caps cut short.</li>
<li>Check that no run stopped at the new cap.</li>
<li>--retest of every other classifier and analysis run: the new metrics, no training.</li>
</ol>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>מטלת אמצע, חלק A: ריצות חוזרות אחרי שינוי תקרת ה-epochs, והמדדים החדשים לכל הריצות</p>
<p>מחברת הרצה דקה: משכפלת את ה-repository, מתקינה אותו ומריצה את train.py. כל הקוד נמצא ב-repository.</p>
<p>למה המחברת הזאת:</p>
<ul>
<li>עם התקרות הישנות (30 epochs ל-linear_probe ול-finetune, ‏12 ל-encoder המפוקח), 18 ריצות נעצרו כשה-val_accuracy עוד עלה. עכשיו התקרה היא 100 לכל מסווג, וה-EarlyStopping (patience 5) עדיין עוצר כל ריצה קודם.</li>
<li>המסווגים רושמים עכשיו גם precision, ‏recall, ‏F1 ו-AUROC (ממוצע על המחלקות), ושומרים את מטריצת הבלבול של ה-test. ה-analysis שומר את אותו הדבר ל-k-NN.</li>
</ul>
<p>שום דבר לא נמחק. ריצה שמאומנת שוב שומרת את התוצאות הישנות שלה בשם חדש (‏results_cap30.json, ‏..._cap12).</p>
<p>סדר התאים:</p>
<ol>
<li>משכפלים את ה-repository, או מושכים את הקוד העדכני.</li>
<li>מתקינים את החבילות ובודקים את ה-GPU (‏L4).</li>
<li>מחברים את Google Drive.</li>
<li>מפעילים את TensorBoard.</li>
<li>22 הריצות שהתקרות הישנות קטעו.</li>
<li>בודקים שאף ריצה לא נעצרה על התקרה החדשה.</li>
<li>‏--retest לכל שאר ריצות המסווגים וה-analysis: המדדים החדשים, בלי אימון.</li>
</ol>
</div>

<div dir="ltr" align="left" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>Step 1: clone the repository</p>
<ul><li>Run it once per runtime. If the folder already exists, git clone fails; run the next cell instead.</li></ul>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>שלב 1: שכפול ה-repository</p>
<ul><li>מריצים פעם אחת בכל סביבת ריצה. אם התיקייה כבר קיימת, git clone נכשל; אז מריצים את התא הבא במקומו.</li></ul>
</div>

In [ ]:
import os

!git clone https://github.com/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR.git
os.chdir("/content/Representation-Learning-Midterm-DAE-SimCLR")
!git log --oneline -1

<div dir="ltr" align="left" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>Step 1 (again): pull the latest code</p>
<ul><li>Instead of the clone cell, when the repository is already on this runtime.</li></ul>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>שלב 1 (שוב): משיכת הקוד העדכני</p>
<ul><li>במקום תא השכפול, כשה-repository כבר נמצא בסביבת הריצה.</li></ul>
</div>

In [ ]:
import os
os.chdir("/content/Representation-Learning-Midterm-DAE-SimCLR")
!git pull
!git log --oneline -1

<div dir="ltr" align="left" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>Step 2: install the requirements and check the GPU</p>
<ul>
<li>Stops with an error if PyTorch sees no GPU. It should be an NVIDIA L4.</li>
<li>Prints the number of CPU cores: config.NUM_WORKERS uses up to 8 of them to load the images.</li>
</ul>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>שלב 2: התקנת החבילות ובדיקת ה-GPU</p>
<ul>
<li>נעצר עם שגיאה אם PyTorch לא רואה GPU. הוא אמור להיות NVIDIA L4.</li>
<li>מדפיס את מספר ליבות ה-CPU: ‏config.NUM_WORKERS משתמש בעד 8 מהן לטעינת התמונות.</li>
</ul>
</div>

In [ ]:
%pip install -q -r requirements.txt

import torch
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → L4 GPU, then rerun this cell."
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB,", os.cpu_count(), "CPU cores")

<div dir="ltr" align="left" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>Step 3: Google Drive</p>
<ul><li>Every run of Part A is in rl_midterm on Drive. The cells below continue or test those runs, in place.</li></ul>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>שלב 3: ‏Google Drive</p>
<ul><li>כל הריצות של חלק A נמצאות ב-rl_midterm ב-Drive. התאים שלמטה ממשיכים או בודקים את הריצות האלה, במקומן.</li></ul>
</div>

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/לימודים/תואר שני/שנה ב/סמסטר א/למידת ייצוגים במודלים עמוקים/מטלות/מטלת אמצע"
OUTPUT_DIR = f"{DRIVE_DIR}/rl_midterm"
TB = f"{OUTPUT_DIR}/tb_logs"
assert os.path.isdir(TB), f"no runs in {TB}"
print(len(os.listdir(TB)), "run folders in", TB)

<div dir="ltr" align="left" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>Step 4: TensorBoard</p>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>שלב 4: ‏TensorBoard</p>
</div>

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{TB}"

<div dir="ltr" align="left" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>Step 5: the 22 runs the old caps cut short</p>
<ul>
<li>RESUME (17 runs): stopped at the cap while val_accuracy was still rising. Their results.json and hparams.yaml get a new name (_cap30, or _cap12 for the supervised encoders), and the same command continues the run from its last.ckpt. With Adam at a fixed learning rate, that is the run it would have been with the new cap.</li>
<li>The supervised ResNet encoders come first. Before they train again, their encoder files are copied to ..._cap12.pt.</li>
<li>FRESH (5 runs): k-NN, linear probe and analysis of the supervised ResNet encoder (all labels). They must not continue from last.ckpt: it holds the OLD encoder. Their whole folder gets the name ..._cap12, and they run from scratch, after that encoder has trained again.</li>
<li>Every rename happens once. After a disconnect, run the cell again: the renames are skipped, unfinished runs continue, finished ones are skipped.</li>
</ul>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>שלב 5: ‏22 הריצות שהתקרות הישנות קטעו</p>
<ul>
<li>‏RESUME (17 ריצות): נעצרו על התקרה כשה-val_accuracy עוד עלה. ה-results.json וה-hparams.yaml שלהן מקבלים שם חדש (‏_cap30, או _cap12 ל-encoders המפוקחים), ואותה פקודה ממשיכה את הריצה מה-last.ckpt שלה. עם Adam וקצב למידה קבוע, זו הריצה שהייתה יוצאת עם התקרה החדשה.</li>
<li>ה-encoders המפוקחים של ResNet רצים ראשונים. לפני שהם מתאמנים שוב, קובצי ה-encoder שלהם מועתקים ל-..._cap12.pt.</li>
<li>‏FRESH (5 ריצות): ‏k-NN, ‏linear probe ו-analysis של ה-encoder המפוקח של ResNet (כל התוויות). אסור שימשיכו מה-last.ckpt: יש בו את ה-encoder הישן. כל התיקייה שלהן מקבלת את השם ..._cap12, והן רצות מאפס, אחרי שה-encoder הזה התאמן שוב.</li>
<li>כל שינוי שם קורה פעם אחת. אחרי ניתוק מריצים את התא שוב: שינויי השמות מדולגים, ריצות שלא הסתיימו ממשיכות, ושהסתיימו מדולגות.</li>
</ul>
</div>

In [ ]:
import shutil

R = "--backbone resnet18"
DAE_R = f"--encoder dae {R} --decoder resnet18"  # the ResNet DAE (stage 2 decoder)
AE_R = f"--encoder dae --noise_level 0 {R} --decoder resnet18"
SUP_R = f"--encoder supervised {R}"
SUP_FILE = "supervised_TinyImageNet_256_{}_resnet18_acc_s42"  # .format(label fraction)

# (run folder, flags). The folder must be the one these flags name; the cell checks it exists.
RESUME = [
    # The supervised ResNet encoders (cap 12). First: the FRESH runs need the 1.0 one.
    *[(f"pretrain_supervise_encoder_{SUP_FILE.format(f)}_TinyImageNet_{f}",
       f"--mode pretrain_supervise_encoder --label_fraction {f} {R}") for f in (0.01, 0.1, 1.0)],
    # Fine-tuning (cap 30)
    *[(f"finetune_dae_TinyImageNet_salt_pepper_{n}_256_resnet18_resnet18dec_s42_{ds}_{f}_acc",
       f"--mode finetune {flags} --dataset {ds.lower()} --label_fraction {f}")
      for n, flags in (("0.5", DAE_R), ("0.0", AE_R)) for ds in ("TinyImageNet", "CIFAR100") for f in (0.01, 0.1)],
    *[(f"finetune_dae_TinyImageNet_salt_pepper_{n}_256_s42_CIFAR100_0.01_acc",
       f"--mode finetune {flags} --dataset cifar100 --label_fraction 0.01")
      for n, flags in (("0.5", "--encoder dae"), ("0.0", "--encoder dae --noise_level 0"))],
    # Linear probes (cap 30)
    ("linear_probe_dae_TinyImageNet_salt_pepper_0.5_256_resnet18_s42_TinyImageNet_1.0_acc",
     f"--mode linear_probe --encoder dae {R}"),  # stage 1: the ResNet DAE with the light decoder
    ("linear_probe_random_resnet18_s42_CIFAR100_1.0_acc", f"--mode linear_probe --encoder random {R} --dataset cifar100"),
    ("linear_probe_random_s42_CIFAR100_1.0_acc", "--mode linear_probe --encoder random --dataset cifar100"),
    ("linear_probe_random_s42_TinyImageNet_1.0_acc", "--mode linear_probe --encoder random"),
]
SUP = SUP_FILE.format(1.0)
FRESH = [
    (f"knn_{SUP}_TinyImageNet_1.0", f"--mode knn {SUP_R}"),
    (f"knn_{SUP}_CIFAR100_1.0", f"--mode knn {SUP_R} --dataset cifar100"),
    (f"linear_probe_{SUP}_TinyImageNet_1.0_acc", f"--mode linear_probe {SUP_R}"),
    (f"linear_probe_{SUP}_CIFAR100_1.0_acc", f"--mode linear_probe {SUP_R} --dataset cifar100"),
    (f"analysis_{SUP}_TinyImageNet_1.0", f"--mode analysis {SUP_R}"),
]
assert len(RESUME) == 17 and len(FRESH) == 5


def move_once(src, dst):
    """Rename src to dst, unless that was done before (dst exists) or src is not there."""
    if os.path.exists(src) and not os.path.exists(dst):
        os.rename(src, dst)
        print("  kept as", os.path.basename(dst))


# Check every folder first, so a wrong name stops the cell before anything changes.
for folder, _ in RESUME:
    assert os.path.isdir(f"{TB}/{folder}"), f"missing run folder: {folder}"
for folder, _ in FRESH:
    assert os.path.isdir(f"{TB}/{folder}") or os.path.isdir(f"{TB}/{folder}_cap12"), f"missing run folder: {folder}"

# Set aside the old results, once.
for folder, flags in RESUME:
    tag = "_cap12" if folder.startswith("pretrain_supervise_encoder") else "_cap30"
    print(folder)
    if not os.path.exists(f"{TB}/{folder}/results{tag}.json"):  # not set aside yet
        move_once(f"{TB}/{folder}/hparams.yaml", f"{TB}/{folder}/hparams{tag}.yaml")
        move_once(f"{TB}/{folder}/results.json", f"{TB}/{folder}/results{tag}.json")
for f in (0.01, 0.1, 1.0):
    enc = f"{OUTPUT_DIR}/encoders/{SUP_FILE.format(f)}.pt"
    if not os.path.exists(enc[:-3] + "_cap12.pt"):
        shutil.copyfile(enc, enc[:-3] + "_cap12.pt")  # a copy: the run overwrites the original when it finishes
        print("  encoder kept as", os.path.basename(enc[:-3] + "_cap12.pt"))
for folder, flags in FRESH:
    move_once(f"{TB}/{folder}", f"{TB}/{folder}_cap12")

for i, (folder, flags) in enumerate(RESUME + FRESH, 1):
    if folder in [r for r, _ in FRESH]:
        # Only after the supervised ResNet encoder (all labels) has finished training again.
        assert os.path.exists(f"{TB}/pretrain_supervise_encoder_{SUP}_TinyImageNet_1.0/results.json"), \
            "the supervised ResNet encoder has not finished: run the cell again"
    print(f"\n=== rerun {i}/{len(RESUME) + len(FRESH)}: {flags}")
    !python train.py {flags} --output_dir "{OUTPUT_DIR}"

<div dir="ltr" align="left" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>Step 6: did any run stop at the new cap?</p>
<ul><li>For each of the 22 runs: the epochs it trained and its best epoch. A run that trained 100 epochs with its best epoch near the end would need a higher cap.</li></ul>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>שלב 6: האם ריצה כלשהי נעצרה על התקרה החדשה?</p>
<ul><li>לכל אחת מ-22 הריצות: כמה epochs התאמנה, ומה ה-epoch הטוב ביותר שלה. ריצה שהתאמנה 100 epochs וה-epoch הטוב ביותר שלה קרוב לסוף הייתה צריכה תקרה גבוהה יותר.</li></ul>
</div>

In [ ]:
import json
import torch
import config

for folder, _ in RESUME + FRESH:
    run = f"{TB}/{folder}"
    with open(f"{run}/results.json") as f:
        results = json.load(f)
    if folder.startswith(("knn_", "analysis_")):
        print(f"{'':>17} test k-NN top-1 {results['test_knn_top1']:.4f}  {folder}")
        continue
    last = torch.load(f"{run}/checkpoints/last.ckpt", map_location="cpu", weights_only=False)
    best = [s["best_model_path"] for k, s in last["callbacks"].items() if k.startswith("ModelCheckpoint") and s.get("monitor")][0]
    best_epoch = int(os.path.basename(best).split("-")[0].split("=")[1]) + 1
    flag = "  <-- stopped at the cap" if last["epoch"] + 1 >= config.TARGET_NUM_EPOCHS else ""
    print(f"epochs {last['epoch'] + 1:3d} best {best_epoch:3d}  test top-1 {results['test_accuracy']:.4f}  {folder}{flag}")

<div dir="ltr" align="left" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>Step 7: the new metrics for every other run (--retest, no training)</p>
<ul>
<li>The command of every classifier and analysis run of Part A: the small encoder (colab_train3.ipynb) and ResNet-18 (colab_resnet.ipynb, stages 1 and 2), each with --retest.</li>
<li>--retest never trains. A classifier run is tested again from its best checkpoint, and the new metrics are added to its results.json. analysis runs again (no training in it either).</li>
<li>The first results.json is kept as results_before_retest.json. If test_accuracy (for analysis: test_knn_top1) comes out different by more than 0.0005, train.py stops with an error and writes nothing; the loop goes on to the next run, and the summary at the end lists it.</li>
<li>A run that already has the new metrics (the 22 of step 5, or a run tested before a disconnect) is skipped. pretrain and knn runs print "already finished".</li>
<li>About a minute per classifier run, about 2 minutes per analysis run.</li>
</ul>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>שלב 7: המדדים החדשים לכל שאר הריצות (‏--retest, בלי אימון)</p>
<ul>
<li>הפקודה של כל ריצת מסווג ו-analysis של חלק A: ה-encoder הקטן (‏colab_train3.ipynb) ו-ResNet-18 (‏colab_resnet.ipynb, שלבים 1 ו-2), כל אחת עם ‎--retest.</li>
<li>‏--retest אף פעם לא מאמן. ריצת מסווג נבדקת שוב מה-checkpoint הטוב ביותר שלה, והמדדים החדשים מתווספים ל-results.json שלה. ה-analysis רץ שוב (גם בו אין אימון).</li>
<li>ה-results.json הראשון נשמר כ-results_before_retest.json. אם test_accuracy (ב-analysis: ‏test_knn_top1) יוצא שונה ביותר מ-0.0005, ‏train.py נעצר עם שגיאה ולא כותב כלום; הלולאה ממשיכה לריצה הבאה, והסיכום בסוף מציג אותה.</li>
<li>ריצה שכבר יש לה את המדדים החדשים (‏22 הריצות של שלב 5, או ריצה שנבדקה לפני ניתוק) מדולגת. ריצות pretrain ו-knn מדפיסות "already finished".</li>
<li>בערך דקה לכל ריצת מסווג, ובערך 2 דקות לכל ריצת analysis.</li>
</ul>
</div>

In [ ]:
FRACTIONS = (0.01, 0.1, 1.0)
DATASETS = ("tinyimagenet", "cifar100")


def protocol_runs(encoders, supervised, extra=""):
    """The classifier and analysis commands of one backbone, as in colab_train3 / colab_resnet."""
    runs = [f"--mode pretrain_supervise_encoder --label_fraction {f}{extra}" for f in FRACTIONS]
    for flags in encoders:
        for dataset in DATASETS:
            runs += [f"--mode linear_probe {flags} --dataset {dataset}",
                     *[f"--mode finetune {flags} --dataset {dataset} --label_fraction {f}" for f in FRACTIONS]]
    runs += [f"--mode linear_probe {supervised} --dataset {dataset}" for dataset in DATASETS]
    runs += [f"--mode analysis {flags}" for flags in (*encoders, supervised)]
    return runs


RETEST_RUNS = (
    # the small encoder (colab_train3.ipynb)
    protocol_runs(["--encoder dae", "--encoder dae --noise_level 0", "--encoder random"], "--encoder supervised")
    # ResNet-18, stage 2 (colab_resnet.ipynb)
    + protocol_runs([DAE_R, AE_R, f"--encoder random {R}"], SUP_R, extra=f" {R}")
    # ResNet-18, stage 1: the DAE with the light decoder
    + [f"--mode linear_probe --encoder dae {R}"]
)
print(len(RETEST_RUNS), "runs")

failed = []
for i, flags in enumerate(RETEST_RUNS, 1):
    print(f"\n=== retest {i}/{len(RETEST_RUNS)}: {flags}")
    output = !python train.py {flags} --retest --output_dir "{OUTPUT_DIR}" 2>&1
    print("\n".join(output[-4:]))  # the last lines: the results, or the error
    if any(line.startswith("Traceback") for line in output):
        failed.append(flags)

print("\nDone.", len(RETEST_RUNS) - len(failed), "OK,", len(failed), "failed:")
print("\n".join(failed))